In [40]:
import duckdb

In [49]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [52]:
df = con.execute("""
    SELECT 
    *, 
    ROW_NUMBER() OVER (PARTITION BY NATBRO ORDER BY data_ingestacao DESC) AS row_num
FROM bronze_Z0019 
WHERE data_ingestacao >= '2025-08-20'
                """).fetchdf()
df.head(10)

,NATBRO,MAKT,WERKS,MAINS,LABST,nome_arquivo,data_ingestacao,row_num
0,10003,CHAVE DE FENDA,B60,100,500,z0019_2.csv,2025-08-20 09:35:18.158888,1
1,10003,CHAVE DE FENDA,B60,100,500,z0019_1.csv,2025-08-20 09:28:36.133547,2
2,10001,PARAFUSO,BT10,100,100,z0019_2.csv,2025-08-20 09:35:18.158888,1
3,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2025-08-20 09:28:36.133547,2
4,10002,MARTELO,BT50,100,2500,z0019_2.csv,2025-08-20 09:35:18.158888,1
5,10002,MARTELO,BT50,100,2500,z0019_1.csv,2025-08-20 09:28:36.133547,2


In [75]:
df = df.rename(columns={'NATBRO': 'id',
                        'MAKT': 'nome_produto',
                        'WERKS': 'descricao_produto',
                        'MAINS': 'valor_produto',
                        'LABST': 'quantidade_produto'})

In [77]:
df.head(10)

,id,nome_produto,descricao_produto,valor_produto,quantidade_produto
0,10003,CHAVE DE FENDA,B60,100,500
1,10003,CHAVE DE FENDA,B60,100,500
2,10001,PARAFUSO,BT10,100,100
3,10001,PARAFUSO,BT10,100,100
4,10002,MARTELO,BT50,100,2500
5,10002,MARTELO,BT50,100,2500


In [81]:
df2 =df.astype({'id': 'int64',
                'nome_produto': 'string',
                'descricao_produto': 'string',
                'valor_produto': 'float64',
                'quantidade_produto': 'int64'})


In [84]:
df2.dtypes

id                             int64
nome_produto          string[python]
descricao_produto     string[python]
valor_produto                float64
quantidade_produto             int64
dtype: object

In [86]:
con.execute("""
        CREATE TABLE IF NOT EXISTS produto (
            id BIGINT,
            nome_produto STRING,
            id_categoria STRING,
            id_fornecedor BIGINT,
            valor_produto FLOAT
            )
""")

In [ ]:
con.execute("INSERT INTO produto SELECT * from df2")
                       

In [91]:
df_resultado = con.execute("SELECT * FROM produto").fetchdf()
df_resultado.head(10)

,id,nome_produto,id_categoria,id_fornecedor,valor_produto
0,10003,CHAVE DE FENDA,B60,100,500.0
1,10003,CHAVE DE FENDA,B60,100,500.0
2,10001,PARAFUSO,BT10,100,100.0
3,10001,PARAFUSO,BT10,100,100.0
4,10002,MARTELO,BT50,100,2500.0
5,10002,MARTELO,BT50,100,2500.0


In [92]:
con.close()